# Fitting J-Lens

In [ ]:
import os
import sys
import json
import importlib
from pathlib import Path

import torch
import transformers

ROOT = Path("/media/am/AM/mi-lens")
CACHE_DIR = ROOT / "tmp" / "hf_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR / "transformers")

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "lenses" / "jacobian_lens"))

import jlens

importlib.reload(jlens)
jlens.configure_logging()

In [ ]:
APERTUS_MUNIN = "danish-foundation-models/munin-apertus-8b"
QWEN_MUNIN = "danish-foundation-models/munin-qwen3.5-9B"

In [ ]:
CACHE_DIR

In [ ]:
MODEL_NAME = APERTUS_MUNIN

tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=str(CACHE_DIR),
)

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    cache_dir=str(CACHE_DIR),
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="auto"
)

In [ ]:
hf_model = hf_model.cpu()

In [ ]:
model = jlens.from_hf(hf_model, tokenizer)
model

In [ ]:
def load_prompt_records(path: Path):
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            records.append(json.loads(line))
    return records


def prompt_text_from_record(record: dict) -> str:
    if "text" in record:
        return str(record["text"])
    if "prompt" in record:
        return str(record["prompt"])
    raise KeyError("Each JSONL row must contain either 'text' or 'prompt'.")


records = load_prompt_records(TRAIN_FIT_PATH)
raw_prompts = [prompt_text_from_record(record) for record in records]

usable_prompts = []
for prompt in raw_prompts:
    token_ids = tokenizer(prompt, truncation=True, max_length=MAX_SEQ_LEN).input_ids
    if len(token_ids) >= 2:
        usable_prompts.append(prompt)

print(f"Loaded {len(records)} records from {TRAIN_FIT_PATH}")
print(f"Usable prompts with >=2 tokens: {len(usable_prompts)}")
usable_prompts[:2]

In [ ]:
lens = jlens.fit(
    model,
    prompts=usable_prompts,
    max_seq_len=MAX_SEQ_LEN,
    checkpoint_path=str(CHECKPOINT_PATH),
    checkpoint_every=CHECKPOINT_EVERY,
)

lens.save(str(LENS_PATH))
lens

In [ ]:
if CHECKPOINT_PATH.exists():
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    print("Checkpoint keys:", sorted(ckpt.keys()))
    print("n_done:", ckpt.get("n_done"))
    print("next_idx:", ckpt.get("next_idx"))
    print("source_layers:", ckpt.get("source_layers"))
    partial_lens = jlens.JacobianLens.load(str(CHECKPOINT_PATH))
    partial_lens
else:
    print("No checkpoint found yet at", CHECKPOINT_PATH)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

layer = lens.source_layers[0]   # or pick any fitted layer
J = lens.jacobians[layer].numpy()

plt.figure(figsize=(8, 6))
sns.heatmap(J, cmap="coolwarm", center=0)
plt.title(f"Jacobian lens matrix at layer {layer}")
plt.xlabel("source residual dim")
plt.ylabel("target residual dim")
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

norm_df = pd.DataFrame({
    "layer": list(lens.jacobians.keys()),
    "fro_norm": [J.norm().item() for J in lens.jacobians.values()],
}).sort_values("layer")

plt.figure(figsize=(10, 2))
sns.heatmap(norm_df[["fro_norm"]].T, cmap="viridis", cbar=True)
plt.xticks(
    ticks=range(len(norm_df)),
    labels=norm_df["layer"],
    rotation=0,
)
plt.yticks([0.5], ["Frobenius norm"], rotation=0)
plt.title("Jacobian strength by layer")
plt.show()

In [ ]:
sns.heatmap(abs(J), cmap="magma")

In [ ]:
import torch
I = torch.eye(J.shape[0]).numpy()
sns.heatmap(J - I, cmap="coolwarm", center=0)

In [ ]:
sns.heatmap(J[:100, :100], cmap="coolwarm", center=0)

In [ ]:
block = J[:100, :100]
lim = abs(block).max()

plt.figure(figsize=(7, 6))
sns.heatmap(block, cmap="coolwarm", center=0, vmin=-lim, vmax=lim)
plt.title(f"Layer {layer} Jacobian block")
plt.show()

In [ ]:
blocks = [lens.jacobians[layer][:100, :100].numpy() for layer in lens.source_layers]
lim = max(abs(block).max() for block in blocks)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, layer, block in zip(axes.flat, lens.source_layers[:6], blocks[:6]):
    sns.heatmap(block, cmap="coolwarm", center=0, vmin=-lim, vmax=lim, ax=ax, cbar=ax is axes.flat[0])
    ax.set_title(f"Layer {layer}")
plt.tight_layout()
plt.show()

In [ ]:
norms = [J.norm().item() for J in lens.jacobians.values()]
norm_df = pd.DataFrame([norms], columns=lens.source_layers)

plt.figure(figsize=(12, 2))
sns.heatmap(norm_df, cmap="viridis", vmin=0, vmax=max(norms), cbar=True)
plt.yticks([0.5], ["Frobenius norm"], rotation=0)
plt.xlabel("Layer")
plt.show()

In [ ]:
import numpy as np

block = J[:100, :100]
lim = np.percentile(np.abs(block), 99)

sns.heatmap(block, cmap="coolwarm", center=0, vmin=-lim, vmax=lim)

# Logit Lens and J-Lens

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt

prompt = "Fact: The currency used in the country shaped like a boot is"
positions = [-2]  # same as your example

layers = lens.source_layers

jl_logits, final_logits, input_ids = lens.apply(
    model,
    prompt,
    layers=layers,
    positions=positions,
    use_jacobian=True,
)

ll_logits, _, _ = lens.apply(
    model,
    prompt,
    layers=layers,
    positions=positions,
    use_jacobian=False,
)

final_top = final_logits[0].argmax().item()
final_top_str = tok.decode([final_top])

rows = []
for layer in layers:
    jl = jl_logits[layer][0]
    ll = ll_logits[layer][0]
    final = final_logits[0]

    rows.append({
        "layer": layer,
        "jl_top": tok.decode([jl.argmax().item()]),
        "ll_top": tok.decode([ll.argmax().item()]),
        "final_top": final_top_str,
        "jl_top_matches_final": int(jl.argmax().item() == final.argmax().item()),
        "ll_top_matches_final": int(ll.argmax().item() == final.argmax().item()),
        "jl_rank_of_final_top": int((jl.argsort(descending=True) == final_top).nonzero()[0].item() + 1),
        "ll_rank_of_final_top": int((ll.argsort(descending=True) == final_top).nonzero()[0].item() + 1),
        "jl_prob_final_top": float(torch.softmax(jl, dim=0)[final_top].item()),
        "ll_prob_final_top": float(torch.softmax(ll, dim=0)[final_top].item()),
    })

df = pd.DataFrame(rows)
df.head(20)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df["layer"], df["jl_rank_of_final_top"], label="Jacobian lens")
plt.plot(df["layer"], df["ll_rank_of_final_top"], label="Logit lens")
plt.yscale("log")
plt.xlabel("Layer")
plt.ylabel("Rank of model-final top token")
plt.title("How early does each lens recover the final prediction?")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df["layer"], df["jl_prob_final_top"], label="Jacobian lens")
plt.plot(df["layer"], df["ll_prob_final_top"], label="Logit lens")
plt.xlabel("Layer")
plt.ylabel("Probability on model-final top token")
plt.title("Probability assigned to the model's final top token")
plt.legend()
plt.show()

In [ ]:
def top_tokens(logits, k=5):
    idx = logits.topk(k).indices.tolist()
    return [tok.decode([i]) for i in idx]

for layer in layers[:: max(1, len(layers)//8)]:
    print(f"Layer {layer}")
    print("  JL:", top_tokens(jl_logits[layer][0]))
    print("  LL:", top_tokens(ll_logits[layer][0]))
    print()

# Logit Lens and J-Lens

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go

def decode_token(token_id):
    return tok.decode([int(token_id)], clean_up_tokenization_spaces=False).replace("\n", "\\n")

def topk_ids_and_tokens(logits, k=5):
    ids = logits.topk(k).indices.tolist()
    toks = [decode_token(i) for i in ids]
    return ids, toks

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a or b) else 1.0

prompt = "Fact: The currency used in the country shaped like a boot is"
top_k = 8

layers = lens.source_layers
positions = list(range(-min(12, model.encode(prompt).shape[1]), 0))

jl_logits, final_logits, input_ids = lens.apply(
    model,
    prompt,
    layers=layers,
    positions=positions,
    use_jacobian=True,
)

ll_logits, _, _ = lens.apply(
    model,
    prompt,
    layers=layers,
    positions=positions,
    use_jacobian=False,
)

seq_tokens = input_ids[0].tolist()
seq_token_strs = [decode_token(t) for t in seq_tokens]
abs_positions = [p if p >= 0 else len(seq_tokens) + p for p in positions]

z = np.zeros((len(layers), len(positions)), dtype=float)
text = np.empty((len(layers), len(positions)), dtype=object)
hover = np.empty((len(layers), len(positions)), dtype=object)

for i, layer in enumerate(layers):
    for j, pos in enumerate(positions):
        jl = jl_logits[layer][j]
        ll = ll_logits[layer][j]
        final = final_logits[j]

        jl_ids, jl_toks = topk_ids_and_tokens(jl, k=top_k)
        ll_ids, ll_toks = topk_ids_and_tokens(ll, k=top_k)
        final_ids, final_toks = topk_ids_and_tokens(final, k=top_k)

        overlap = jaccard(jl_ids, ll_ids)
        top1_match = jl_ids[0] == ll_ids[0]
        inter = [decode_token(t) for t in sorted(set(jl_ids) & set(ll_ids))]

        z[i, j] = overlap
        text[i, j] = f"{jl_toks[0]} | {ll_toks[0]}"
        hover[i, j] = (
            f"Layer: {layer}<br>"
            f"Position: {abs_positions[j]}<br>"
            f"Prompt token: {seq_token_strs[abs_positions[j]]}<br>"
            f"JL top-1: {jl_toks[0]!r}<br>"
            f"LL top-1: {ll_toks[0]!r}<br>"
            f"Top-1 match: {top1_match}<br>"
            f"Jaccard@{top_k}: {overlap:.3f}<br>"
            f"Intersection: {inter}<br><br>"
            f"JL top-{top_k}: {jl_toks}<br>"
            f"LL top-{top_k}: {ll_toks}<br>"
            f"Final top-{top_k}: {final_toks}"
        )

fig = go.Figure(
    data=go.Heatmap(
        z=z,
        x=abs_positions,
        y=layers,
        text=text,
        customdata=hover,
        hovertemplate="%{customdata}<extra></extra>",
        texttemplate="%{text}",
        textfont={"size": 10},
        colorscale="Viridis",
        zmin=0,
        zmax=1,
        colorbar={"title": f"Jaccard@{top_k}"},
    )
)

fig.update_layout(
    width=1200,
    height=700,
    title=f"Jacobian Lens vs Logit Lens overlap on top-{top_k} predictions",
    xaxis_title="Prompt position",
    yaxis_title="Layer",
)

fig.show()